## Machine Learning Pipeline

As the class practice, the students will be required to develop a machine learning pipeline using the `Churn_Modelling_train_test.csv` dataset.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)


### Exploratory Data Analysis

In this section the students are required to do an EDA to understand the dataset.

In [300]:
# import packages
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

In [301]:
# Load the dataset
df = pd.read_csv("/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/session2/datasets/Churn_Modelling_train_test.csv")

In [302]:
# dataset analysis
df.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,4784,15729224,Jennings,710,France,Female,37.0,5,0.00,2,1.0,0.0,115403.31,0
1,1497,15799156,Okwuadigbo,569,Spain,Male,38.0,8,0.00,2,0.0,0.0,79618.79,0
2,1958,15674922,Beavers,710,France,Male,54.0,6,171137.62,1,1.0,1.0,167023.95,1
3,9174,15653572,Thornton,673,Spain,Male,43.0,8,127132.96,1,0.0,1.0,6009.27,1
4,9748,15775761,Iweobiegbunam,610,Germany,Female,69.0,5,86038.21,3,0.0,0.0,192743.06,1


In [303]:
# review categorical variables
catgorical_variables = df.select_dtypes(include="object").columns.drop('Gender')

for col in catgorical_variables:
    print(df[col].value_counts())


Surname
Smith          29
Yeh            25
Martin         25
Walker         24
Genovese       23
               ..
Beit            1
Onwuamaegbu     1
Edith           1
Holder          1
Macrossan       1
Name: count, Length: 2772, dtype: int64
Geography
France     4505
Spain      2250
Germany    2245
Name: count, dtype: int64


In [304]:

#preprocess binary variables
df.loc[df['HasCrCard'] == 1.0, 'HasCrCard'] = 1
df.loc[df["HasCrCard"] == 0.0, "HasCrCard"] = 0

df.loc[df['IsActiveMember'] == 1.0, 'IsActiveMember'] = 1
df.loc[df["IsActiveMember"] == 0.0, "IsActiveMember"] = 0

df.loc[df['Exited'] == 1.0, 'Exited'] = 1
df.loc[df["Exited"] == 0.0, "Exited"] = 0

df.loc[df['Gender'] == 'Female', 'Gender'] = 1
df.loc[df["Gender"] == 'Male', "Gender"] = 0
# review binary variables
binary_variables = ['HasCrCard', 'IsActiveMember', 'Exited','Gender']
for col in binary_variables:
    print(df[col].value_counts())

HasCrCard
1.0    6351
0.0    2649
Name: count, dtype: int64
IsActiveMember
1.0    4633
0.0    4368
Name: count, dtype: int64
Exited
0    7161
1    1840
Name: count, dtype: int64
Gender
0    4906
1    4095
Name: count, dtype: int64


In [305]:
# review numerical variables
numerical_variables = df.select_dtypes(include=["int64", "float64"]).columns.difference(binary_variables)
for col in numerical_variables:
     print(df[col].describe())

count    9000.000000
mean       38.901781
std        10.450760
min        18.000000
25%        32.000000
50%        37.000000
75%        44.000000
max        92.000000
Name: Age, dtype: float64
count      9001.000000
mean      76222.210827
std       62432.198151
min           0.000000
25%           0.000000
50%       96997.090000
75%      127450.140000
max      250898.090000
Name: Balance, dtype: float64
count    9001.000000
mean      650.681369
std        96.539591
min       350.000000
25%       584.000000
50%       652.000000
75%       718.000000
max       850.000000
Name: CreditScore, dtype: float64
count    9.001000e+03
mean     1.569072e+07
std      7.179353e+04
min      1.556570e+07
25%      1.562811e+07
50%      1.569096e+07
75%      1.575265e+07
max      1.581569e+07
Name: CustomerId, dtype: float64
count      9001.000000
mean     100180.823967
std       57561.189534
min          11.580000
25%       50972.600000
50%      100556.980000
75%      149458.730000
max      199992.4800

In [306]:
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,9001.000000,9.001000e+03,9001.000000,9000.000000,9001.000000,9001.000000,9001.000000,9000.000000,9001.000000,9001.000000,9001.000000
mean,5003.366515,1.569072e+07,650.681369,38.901781,5.007888,76222.210827,1.531497,0.705667,0.514721,100180.823967,0.204422
std,2884.787499,7.179353e+04,96.539591,10.450760,2.894025,62432.198151,0.579398,0.455768,0.499811,57561.189534,0.403301
min,2.000000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,11.580000,0.000000
25%,2509.000000,1.562811e+07,584.000000,32.000000,2.000000,0.000000,1.000000,0.000000,0.000000,50972.600000,0.000000
50%,4989.000000,1.569096e+07,652.000000,37.000000,5.000000,96997.090000,1.000000,1.000000,1.000000,100556.980000,0.000000
75%,7505.000000,1.575265e+07,718.000000,44.000000,7.000000,127450.140000,2.000000,1.000000,1.000000,149458.730000,0.000000
max,10000.000000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.000000,1.000000,199992.480000,1.000000


**EDA Conclusions**

From this Exploratory Data Analysis we can retrieve the following information:
* *To be completed by the student*

### ML Pipeline

In this section the students are required to create a ML pipeline to predict whether the customer left the bank or not.

In [307]:
# Load the dataset
df = pd.read_csv("/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/session2/datasets/Churn_Modelling_train_test.csv")

In [308]:
#Preprocess 
df = df.dropna()

df.loc[df['Gender'] == 'Female', 'Gender'] = 1
df.loc[df["Gender"] == 'Male', "Gender"] = 0
df["Gender"] = df["Gender"].astype(int)

df["IsActiveMember"] = df["IsActiveMember"].astype(int)

df["HasCrCard"] = df["HasCrCard"].astype(int)

df= df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])


In [309]:
# Preprocess the features - create a function or a class for it 
def transform(df: pd.DataFrame) -> pd.DataFrame:
    pass

In [310]:
# balance the dataset (if you think it is necessary)
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Separate the classes
    df_y0 = df[df['Exited'] == 0]
    df_y1 = df[df['Exited'] == 1]

    # Find the smaller class size
    min_size = len(df_y1)

    # Randomly sample from each class
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    # Concatenate back together
    df_balanced = pd.concat([df_y0_balanced, df_y1])

    # Shuffle the dataset
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

In [311]:
# Train a decision tree model using the `train sample`
df.loc[df["Exited"]] = 1
df.loc[df["Exited"]] = 0
df["Exited"] = df["Exited"].astype(int)

In [312]:
df['Geography'] = df['Geography'].astype(str)

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df[['Geography']])
encoded_df = encoder.transform(df[['Geography']])

df = pd.concat([df.drop(columns=['Geography']), encoded_df], axis=1)


In [313]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,0,0,0.0,0,0.00,0,0,0,0.00,0,0.0,0.0,0.0
1,0,0,0.0,0,0.00,0,0,0,0.00,0,0.0,0.0,0.0
2,749,0,47.0,9,110022.74,1,0,1,135655.29,1,0.0,1.0,0.0
3,724,0,34.0,6,118235.70,2,0,0,157137.23,0,0.0,1.0,0.0
4,586,1,46.0,0,0.00,3,0,1,131553.82,1,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3675,592,1,34.0,9,0.00,2,1,1,20460.20,0,1.0,0.0,0.0
3676,608,0,33.0,1,102772.67,2,1,0,70705.58,0,0.0,1.0,0.0
3677,476,1,40.0,4,0.00,2,0,0,182547.04,0,1.0,0.0,0.0
3678,453,0,29.0,6,0.00,1,0,0,198376.02,1,1.0,0.0,0.0


Train And Evaluate model

In [314]:
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [315]:
# import packages for mlfow
import mlflow
from mlflow.models import infer_signature

Start a local Tracking server on port 8080 - it is necessary to import mlflow and open a new terminal

Visit the MLFlow UI on `http://127.0.0.1:8080`

In [ ]:


mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

2026/05/19 18:13:27 INFO mlflow.tracking.fluent: Experiment with name 'Practice Experiment - Juan Carlos Vilar' does not exist. Creating a new experiment.


In [320]:
# Create a new MLflow Experiment called `Practice Experiment - {Your Name}
mlflow.set_experiment("Practice Experiment - Juan Carlos Vilar")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
with mlflow.start_run():
    # Log the hyperparameters
    params = {
    "n_estimators": 100, # number of trees     
    "max_depth": 10, # mumber of levels to avoid overfitting    
    "min_samples_split": 2,# min samples to split
    "min_samples_leaf": 1, # min leaf node
    "max_features": "sqrt", # standard for classification
    "bootstrap": True, # sample with replacement
    "random_state": 73, 
    "n_jobs": -1, #also stadard for random forest (all cores)
}
    mlflow.log_params(params)

    # Train and get predictions
    rf = RandomForestClassifier(**params)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    # Log the loss metric
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    f1= f1_score(y_test, y_pred)
    mlflow.log_metric("f1_score", f1)

    # Set a tag
    mlflow.set_tag("model_info", "Random Forest Classifier for Churn Prediction")

    # Infer the model signature
    signature = mlflow.models.infer_signature(X_train, rf.predict(X_train))

    # Log the model
    mlflow.sklearn.log_model(rf, "random_forest_model", signature=signature)

/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Python(49441) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


🏃 View run rebellious-yak-453 at: http://127.0.0.1:8080/#/experiments/274289617418714620/runs/d792f1f1d37849f4bffef86e82147e2c
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/274289617418714620


### Keep practicing

I created 2 test with different parameters, they were saved on mlflow in Churn experiment.

Paramenters changed are the Number of trees and layers. I got similar results, the model are: 
- rebellious-yak-453 --> 100 trees , 10 layers
- thoughtful-fox-692 --> 2000 trees , 20 layers

Now I'll do a logistic regression. Let's see how it goes


In [321]:
mlflow.set_experiment("Practice Experiment - Juan Carlos Vilar")
from sklearn.metrics import accuracy_score
with mlflow.start_run():
    # Log the hyperparameters
    params = {

    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,

    }       
    mlflow.log_params(params)

    # Train and get predictions
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)
    # Log the loss metric
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("f1_score", f1)

    # Set a tag
    mlflow.set_tag("model_info", "Logistic Regression for Churn Prediction")

    # Infer the model signature
    signature = mlflow.models.infer_signature(X_train, rf.predict(X_train))

    # Log the model
    mlflow.sklearn.log_model(rf, "random_forest_model", signature=signature)

/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/.venv/lib/python3.12/s

🏃 View run gregarious-lark-248 at: http://127.0.0.1:8080/#/experiments/274289617418714620/runs/213a805fd96e42b7b7b3d3c5babaf805
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/274289617418714620


It performed worse... with and accuracy of .69 and f1 socore of .70. For the other 2 models I got accuracy 0.77 in both and f1 socore of .77 and .78. 